# Filament PQ: Two Experts Agree at 0.34

This competition scores instance masks with Panoptic Quality (PQ) against filaments that experts outlined by hand, and 296 of the 707 training images were outlined independently by two or three of them. So one question can be answered from the training labels alone: what PQ does one expert get when scored against another, with the host's own scoring logic? \_.tushhaaarrrr's pipeline notebook already runs this check at 1,024 px and reports about 0.34 (credits at the end). Here I measure it at full resolution with the host's exact logic, break it down, and turn it into ways to read a score: how far a score moves by chance, when one more predicted filament is worth keeping, and what turning a mask into instances costs.

Every measured number in the text is printed by a cell below. Apart from one crop of one training image with the experts' outlines (section 3), every output is an aggregate: no masks, polygons or per-filament rows are written out. To score your own out-of-fold predictions the same way, set `MY_PREDICTIONS` in section 7 and run the notebook.

## The numbers to read a score by

| reference point | value | what it means |
|---|---|---|
| One expert scored against another: pooled PQ with the host's logic, 296 images | **0.340** | how closely two people who outline the same image agree on this metric |
| SQ of their matches (mean IoU) | 0.634 | even when both outline the same filament, the outlines differ a lot |
| Their match rate at IoU > 0.5 | 54% | about half the time, a filament one expert outlined has a match in the other's set |
| SD of one fixed model's PQ over a random 90-image half of the test set, estimated on training images | 0.014 | the public score alone moves by about ±0.03 (95% of the time) with the image draw |
| Public leaderboard on October 12, 2026: median, 75th and 90th percentile of 972 teams | 0.34, 0.37, 0.40 | where the field sat on that date |

How I read a score with these:

- **0.34** is what one expert scores against another, and the median team on the board.
- **0.40** is above single-expert agreement by more than the noise, at the board's 90th percentile on October 12, 2026.
- Within the noise of a 90-image half (section 4) are gaps under about 0.03 between a local CV score and the public board, under about 0.04 between the public and private boards, and under about 0.02 between two public scores, before the board's rounding to two decimals.

## 1. The labels as a ground-truth table

The training labels are one COCO JSON: a polygon per filament, and an "image" record per annotator and image whose id is `<annotator>-<stem>`. The next cell finds that file by its content, because the mount path differs between notebook setups, and turns it into the table the host's scorer reads: one row per filament, with `filament_id = <annotator>-<stem>_<k>` and the RLE string a submission would carry.

In [ ]:
import collections, glob, itertools, json, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image
from scipy import ndimage as ndi
from pycocotools import mask as mu

try:
    get_ipython().run_line_magic("matplotlib", "inline")    # older matplotlib shows the first figure as text without it
except NameError:
    pass
T0 = time.time()
INPUT = os.environ.get("FILAMENT_INPUT", "/kaggle/input")   # set FILAMENT_INPUT to run this outside Kaggle
H = W = 2048                                                # every frame is 2048 x 2048 pixels
BLUE, ORANGE, AQUA, GRAY, INK, INK2 = "#2a78d6", "#eb6834", "#1baf7a", "#c3c2b7", "#0b0b0b", "#52514e"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": GRAY,
                     "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 11, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
                     "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
                     "legend.frameon": False, "font.size": 9.5})


def find_train_labels(root):
    """The training labels, found by content because the mount path differs between notebook setups:
    the COCO JSON whose categories have the supercategory 'filament'."""
    for path in sorted(glob.glob(os.path.join(root, "**", "*.json"), recursive=True)):
        try:
            with open(path) as f:
                coco = json.load(f)
        except (OSError, ValueError):
            continue
        if (isinstance(coco, dict) and {"images", "annotations", "categories"} <= set(coco)
                and any(c.get("supercategory") == "filament" for c in coco["categories"])):
            return path, coco
    raise FileNotFoundError(f"no COCO JSON with filament categories under {root}: add the competition data")


def coco_to_gt_df(coco):
    """COCO JSON -> one row per filament in the host's ground-truth format (filament_id, segmentation_rle), plus
    helper columns. filament_id is '<annotator>-<stem>_<k>': the host's code splits it on the first '_' to get the
    annotator-set and splits that on the first '-' to get the image."""
    stem_of = {im["id"]: os.path.splitext(im["file_name"])[0] for im in coco["images"]}
    k, rows = collections.Counter(), []
    for a in coco["annotations"]:
        aset = a["image_id"]
        k[aset] += 1
        rle = mu.merge(mu.frPyObjects(a["segmentation"], H, W))
        rows.append((f"{aset}_{k[aset]}", rle["counts"].decode("ascii"), aset, stem_of[aset],
                     a["category_id"], int(mu.area(rle)), a["area"]))
    return pd.DataFrame(rows, columns=["filament_id", "segmentation_rle", "annotator_set", "stem",
                                       "category_id", "area", "json_area"])


LABELS, COCO = find_train_labels(INPUT)
GT = coco_to_gt_df(COCO)
SETS_OF = collections.defaultdict(list)                     # image stem -> its annotator-sets, in the JSON's order
for im in COCO["images"]:
    SETS_OF[os.path.splitext(im["file_name"])[0]].append(im["id"])
CATEGORY = {c["id"]: c["name"] for c in COCO["categories"]}
IMAGE = {os.path.splitext(os.path.basename(p))[0]: p
         for p in glob.glob(os.path.join(INPUT, "**", "*.jp*g"), recursive=True)}
TEST_STEMS = sorted(s for s in IMAGE if s not in SETS_OF)   # the images without labels are the test images

per_image = collections.Counter(len(v) for v in SETS_OF.values())
print("labels:", LABELS)
print(f"{len(SETS_OF)} training images, {len(COCO['images'])} annotator-sets, {len(GT)} filaments")
print("images by number of annotator-sets: "
      + ", ".join(f"{k} set{'s' if k > 1 else ''} {per_image[k]}" for k in sorted(per_image)))
print("filaments by category: "
      + ", ".join(f"{CATEGORY[c]} {n}" for c, n in sorted(collections.Counter(GT["category_id"]).items())))
print(f"pycocotools area equals the JSON area for {int((GT['area'] == GT['json_area']).sum())} of {len(GT)} filaments")
print(f"image files: {sum(s in IMAGE for s in SETS_OF)} training, {len(TEST_STEMS)} test")

The pycocotools area of every polygon equals the area stored in the JSON (8,199 of 8,199), so the rasterised masks have exactly the areas the labels record. 411 images have one annotator-set, 145 have two and 151 have three.

## 2. The host's scorer, in a few lines

The host shared the reference scorer as [Self_Evaluation_Notebook](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook) by Azim Ahmadzadeh, and when asked in the forum how the leaderboard score is calculated, pointed to it as the exact implementation of PQ ([topic](https://www.kaggle.com/competitions/filament-segmentation-2026/discussion/727186)). Reading its code, the rules are:

1. Ground-truth filaments are grouped by annotator-set (`<annotator>-<stem>`) and predictions by image (`<stem>`). Each annotator-set is compared with all the predictions for its image, so an image labelled by three people is scored three times.
2. A (filament, prediction) pair is a hit when its IoU is above 0.5, strictly.
3. Every hit pair is a true positive; a prediction that hits nothing is a false positive, and a filament that nothing hits is a false negative.
4. The IoUs and counts are pooled over all annotator-sets and images into one PQ = (sum of TP IoUs) / (TP + 0.5 FP + 0.5 FN). There is no per-image or per-class averaging.

Rule 3 counts every hit pair instead of a one-to-one matching. With pixel-disjoint predictions a filament can be hit by at most one prediction, so for a valid, pixel-disjoint file the rule gives the same score as a one-to-one matching (up to the rare labels that overlap each other); the host has said that overlapping predictions are not accepted ([topic](https://www.kaggle.com/competitions/filament-segmentation-2026/discussion/735377)).

Below is the same logic on the RLE strings with pycocotools, so no 2048 x 2048 mask is ever built. It also returns SQ (the mean IoU of the true positives) and RQ (TP / (TP + 0.5 FP + 0.5 FN)), because PQ = SQ x RQ and the two parts tell different stories.

In [ ]:
def rles(strings):
    """Submission strings -> pycocotools RLE objects. The size is fixed at 2048 x 2048 and never stored."""
    return [{"size": [H, W], "counts": s.encode("ascii")} for s in strings]


def mask_to_rle(mask):
    """(2048, 2048) 0/1 array -> submission string. COCO counts run down the columns, hence np.asfortranarray."""
    assert mask.shape == (H, W), mask.shape
    return mu.encode(np.asfortranarray(mask.astype(np.uint8)))["counts"].decode("ascii")


def pq_counts(gt_df, pred_df, thr=0.5):
    """The host's scoring logic on RLE strings: (sum of TP IoUs, TP, FP, FN), pooled over every annotator-set.
    gt_df.filament_id is '<annotator>-<stem>_<k>'; pred_df.filament_id is '<stem>_<k>'."""
    by_image = {}
    if len(pred_df):
        stem = pred_df["filament_id"].str.split("_", n=1).str[0]
        by_image = {s: rles(g) for s, g in pred_df["segmentation_rle"].groupby(stem)}
    aset = gt_df["filament_id"].str.split("_", n=1).str[0]
    iou_sum, tp, fp, fn = 0.0, 0, 0, 0
    for a, g in gt_df["segmentation_rle"].groupby(aset, sort=False):   # one comparison per annotator-set,
        gt, pred = rles(g), by_image.get(a.split("-", 1)[1], [])        # against every prediction for its image
        if not pred:
            fn += len(gt)
            continue
        iou = np.asarray(mu.iou(pred, gt, [0] * len(gt))).T             # (n_gt, n_pred), the host's iou_matrix
        hit = iou > thr                                                  # strictly above 0.5
        iou_sum += float(iou[hit].sum())
        tp += int(hit.sum())                                             # every hit pair is a true positive
        fp += int((hit.sum(0) == 0).sum())                               # predictions that hit nothing
        fn += int((hit.sum(1) == 0).sum())                               # filaments that nothing hits
    return iou_sum, tp, fp, fn


def summarize(iou_sum, tp, fp, fn):
    den = tp + 0.5 * fp + 0.5 * fn
    return {"PQ": iou_sum / den if den else 0.0, "SQ": iou_sum / tp if tp else 0.0,
            "RQ": tp / den if den else 0.0, "TP": int(tp), "FP": int(fp), "FN": int(fn)}


def host_pq(gt_df, pred_df):
    """Pooled PQ, SQ and RQ the way the leaderboard computes PQ (PQ = SQ x RQ)."""
    return summarize(*pq_counts(gt_df, pred_df))


def as_prediction(rows):
    """Label rows renamed as a prediction for their image: '<annotator>-<stem>_<k>' -> '<stem>_<k>'."""
    return pd.DataFrame({"filament_id": rows["filament_id"].str.split("-", n=1).str[1].to_numpy(),
                         "segmentation_rle": rows["segmentation_rle"].to_numpy()})


def show(r):
    return f"PQ {r['PQ']:.4f} (SQ {r['SQ']:.4f}, RQ {r['RQ']:.4f}; TP {r['TP']}, FP {r['FP']}, FN {r['FN']})"

Four checks before I trust it: labels scored against themselves, no predictions at all, the classic encoding mistake, and the host's own way of computing IoU.

In [ ]:
single = [v[0] for v in SETS_OF.values() if len(v) == 1]           # images labelled by one annotator
own = GT[GT["annotator_set"].isin(single)]
r = host_pq(own, as_prediction(own))
assert r["PQ"] == 1.0 and r["FP"] == r["FN"] == 0
print(f"1. labels scored against themselves ({len(single)} single-annotator images): {show(r)}")

r = host_pq(own, as_prediction(own.iloc[:0]))
assert r["PQ"] == 0.0 and r["FN"] == len(own)
print(f"2. no predictions at all: {show(r)}")

few = own[own["stem"].isin(sorted(set(own["stem"]))[:40])]
flipped = few.assign(segmentation_rle=[mask_to_rle(mu.decode(x).T) for x in rles(few["segmentation_rle"])])
r = host_pq(few, as_prediction(flipped))
assert r["PQ"] < 0.05
print(f"3. the same labels with every mask transposed, as a row-major encoder writes them "
      f"({few['stem'].nunique()} images): {show(r)}")


def dense(strings):
    """Masks the way the host's notebook holds them: one float32 row of H * W pixels per mask."""
    m = mu.decode(rles(strings))                                     # (H, W, n), column-major
    return m.reshape(H * W, -1, order="F").T.astype(np.float32)


pairs = [v for v in SETS_OF.values() if len(v) == 2][:12]
dense_counts = np.zeros(4)
for a, b in pairs:                                                   # a predicts, b is the ground truth
    g = dense(GT.loc[GT["annotator_set"] == b, "segmentation_rle"])
    p = dense(GT.loc[GT["annotator_set"] == a, "segmentation_rle"])
    inter = g @ p.T
    union = g.sum(1)[:, None] + p.sum(1)[None, :] - inter
    iou = np.where(union == 0, 0.0, inter / np.maximum(union, 1))    # the host's torch.where, in numpy
    hit = iou > 0.5
    dense_counts += [iou[hit].sum(), hit.sum(), (hit.sum(0) == 0).sum(), (hit.sum(1) == 0).sum()]
rle_counts = pq_counts(GT[GT["annotator_set"].isin([b for a, b in pairs])],
                       as_prediction(GT[GT["annotator_set"].isin([a for a, b in pairs])]))
same = [int(x) for x in dense_counts[1:]] == list(rle_counts[1:])
assert same and abs(summarize(*dense_counts)["PQ"] - summarize(*rle_counts)["PQ"]) < 1e-5
print(f"4. IoU from dense float32 masks (the host's computation) against IoU from RLEs, {len(pairs)} two-annotator "
      f"images: TP, FP and FN identical ({same}); PQ {summarize(*dense_counts)['PQ']:.6f} and "
      f"{summarize(*rle_counts)['PQ']:.6f}")

The third check is the one to worry about in a pipeline: a mask encoded row by row instead of column by column decodes to its own transpose, which is still a valid 2048 x 2048 mask, so nothing in the file format flags it and PQ falls to 0.0000. Encoding with `np.asfortranarray`, as `mask_to_rle` does, avoids it. The fourth check recomputes IoU the way the host's notebook does, from dense float32 masks with intersections as a matrix product, and gets identical counts.

## 3. How well do two experts agree?

The Overview says the filaments "have been identified by expert human annotators". To put a number on their agreement, I score each annotator's filaments as if they were a submission for that image, with the function above, against the image's other annotators, and rotate through every annotator. That is the comparison a model faces, with a person in the model's place.

In [ ]:
R = {a: rles(g) for a, g in GT.groupby("annotator_set", sort=False)["segmentation_rle"]}
AREA = {a: g.to_numpy() for a, g in GT.groupby("annotator_set", sort=False)["area"]}
CAT = {a: g.to_numpy() for a, g in GT.groupby("annotator_set", sort=False)["category_id"]}
MULTI = [s for s, v in SETS_OF.items() if len(v) > 1]              # images with 2 or 3 annotator-sets


def human_vs_human(stems, min_area=0, table=None):
    """Every annotator of every image in `stems` scored with the host's logic against that image's other
    annotators, pooled. Round r takes the r-th annotator of each image as the prediction, so each image has one
    prediction set per round, as in a submission. Predicted filaments under min_area pixels are dropped; the
    ground truth is never touched. `table` swaps in other predictions with the same columns (section 6)."""
    table = GT if table is None else table
    total = np.zeros(4)
    for r in range(3):
        images = [s for s in stems if len(SETS_OF[s]) > r]
        users = [SETS_OF[s][r] for s in images]
        pred = table[table["annotator_set"].isin(users) & (table["area"] >= min_area)]
        truth = GT[GT["stem"].isin(images) & ~GT["annotator_set"].isin(users)]
        total += pq_counts(truth, as_prediction(pred))
    return summarize(*total)


PAIRS = []        # every ordered pair of annotators on one image: (stem, n_sets, predictor, truth, IoU (n_gt, n_pred))
for s in MULTI:
    for a, b in itertools.permutations(SETS_OF[s], 2):
        PAIRS.append((s, len(SETS_OF[s]), a, b, np.asarray(mu.iou(R[a], R[b], [0] * len(R[b]))).T))

HVH = human_vs_human(MULTI)
n_gt = sum(iou.shape[0] for *_, iou in PAIRS)
matched = sum(int((iou > 0.5).any(1).sum()) for *_, iou in PAIRS)
print(f"one annotator scored against the others: {len(MULTI)} images with 2 or 3 annotator-sets, "
      f"{len(PAIRS)} ordered pairs")
print(f"  all images: {show(HVH)}")
BY_SIZE = {}
for k in (2, 3):
    part = [s for s in MULTI if len(SETS_OF[s]) == k]
    BY_SIZE[k] = human_vs_human(part)
    print(f"  {k} annotator-sets ({len(part)} images): {show(BY_SIZE[k])}")
# Is that gap more than chance? Resample images within each group (all ordered pairs of an image travel together).
per_img = collections.defaultdict(lambda: np.zeros(4))
for s, k, a, b, iou in PAIRS:
    hit = iou > 0.5
    per_img[s] += [iou[hit].sum(), hit.sum(), (hit.sum(0) == 0).sum(), (hit.sum(1) == 0).sum()]
groups = {k: np.array([per_img[s] for s in MULTI if len(SETS_OF[s]) == k]) for k in (2, 3)}
assert all(abs(summarize(*groups[k].sum(0))["PQ"] - BY_SIZE[k]["PQ"]) < 1e-9 for k in (2, 3))
rs = np.random.RandomState(1)
gaps = [summarize(*groups[2][rs.randint(0, len(groups[2]), len(groups[2]))].sum(0))["PQ"]
        - summarize(*groups[3][rs.randint(0, len(groups[3]), len(groups[3]))].sum(0))["PQ"] for _ in range(2000)]
print(f"  gap between them {BY_SIZE[2]['PQ'] - BY_SIZE[3]['PQ']:.4f}, bootstrap SD {np.std(gaps):.4f} "
      f"(images resampled within each group)")
print(f"  match rate: {matched} of {n_gt} filament comparisons ({matched / n_gt:.1%}) find a match (IoU > 0.5) "
      f"in the other annotator's set")
unmatched = [iou[~(iou > 0.5).any(1)] for *_, iou in PAIRS]           # IoU rows of the comparisons without a match
n_un = sum(u.shape[0] for u in unmatched)
touching = sum(int((u > 0).any(1).sum()) for u in unmatched)
close = sum(int((u > 0.2).any(1).sum()) for u in unmatched)
print(f"  of the {n_un} without a match, {touching} ({touching / n_un:.1%}) share pixels with a filament of the other "
      f"annotator, {close} ({close / n_un:.1%}) overlap one at IoU above 0.2, and {n_un - touching} "
      f"({1 - touching / n_un:.1%}) touch none of them")

One expert's outlines score **0.3399** against another's. RQ (0.536) is lower than SQ (0.634): only 53.6% of the comparisons find a match at IoU > 0.5, and the matched outlines overlap at IoU 0.634 on average. Of the comparisons without a match, 61.7% share pixels with one of the other annotator's outlines and 52.5% overlap one at IoU above 0.2 (but not above 0.5); only 38.3% touch nothing the other annotator drew. So more than half of the misses are outlines that disagree too much, not filaments one expert skipped. Images with two annotator-sets score 0.3596 and images with three 0.3334, but that gap is only 1.7 bootstrap SDs, so I would not read much into it.

Where the disagreement sits, split by the labelled filament's area and by category. A filament on a three-annotator image is compared with each of the two other annotators, so the counts are comparisons. Each true positive, false negative and IoU is filed under the labelled filament and each false positive under the predicted one, so the rows add up exactly to the pooled counts above.

In [ ]:
def decompose(group_of):
    """Split PQ's numerator and denominator by a filament property. TP, FN and IoU go to the labelled filament's
    group and FP to the predicted filament's group, so the groups add up to the pooled counts. A filament on a
    three-annotator image is compared with each of the two other annotators, so the counts are comparisons."""
    acc = collections.defaultdict(lambda: np.zeros(6))   # IoU sum, TP, FP, FN, comparisons, matched ones
    for s, k, a, b, iou in PAIRS:
        hit = iou > 0.5
        for grp, h, v in zip(group_of(b), hit, iou):
            acc[grp] += [v[h].sum(), h.sum(), 0, not h.any(), 1, h.any()]
        for grp, h in zip(group_of(a), hit.T):
            acc[grp][2] += not h.any()
    return dict(acc)


def report(title, acc, keys, label):
    print(f"\n{title:<26}{'comparisons':>12}{'match rate':>12}{'SQ':>8}{'PQ':>8}")
    for key in keys:
        if key in acc:
            iou_sum, tp, fp, fn, n, m = acc[key]
            r = summarize(iou_sum, tp, fp, fn)
            print(f"{label(key):<26}{int(n):>12}{m / n:>12.1%}{r['SQ']:>8.3f}{r['PQ']:>8.3f}")
    total = sum(acc.values())
    assert np.allclose(total[:4], [HVH["SQ"] * HVH["TP"], HVH["TP"], HVH["FP"], HVH["FN"]])


EDGES = [0, 200, 400, 800, 1600, 3200, 6400, np.inf]
BIN_NAMES = (["under 200 px"] + [f"{lo:,}-{hi:,} px" for lo, hi in zip(EDGES[1:-2], EDGES[2:-1])]
             + ["6,400 px and up"])
BY_AREA = decompose(lambda a: np.digitize(AREA[a], EDGES) - 1)
BY_CAT = decompose(lambda a: CAT[a])
report("labelled filament area", BY_AREA, range(len(BIN_NAMES)), lambda i: BIN_NAMES[i])
report("category", BY_CAT, sorted(CATEGORY), lambda c: CATEGORY[c])
empty = [CATEGORY[c] for c in sorted(CATEGORY) if c not in BY_CAT]
print(f"\nboth tables add up to the pooled counts; categories with no labelled filaments: {', '.join(empty) or 'none'}")

Small filaments are where agreement collapses: under 200 px only 22% of the comparisons find a match (PQ 0.137, on 58 comparisons), against 63% for filaments of 3,200-6,400 px. Filaments labelled Unidentifiable are matched less often (44%) than Left (58%) or Right (61%) ones.

In this crop of one three-annotator training image, the three outlines sit on the same filaments but differ in how far each one extends, and one small filament is outlined by only two of the three experts.

In [ ]:
STEM, Y0, X0, S = "20110306082634Lh", 200, 1030, 520    # one 520 x 520 crop of a three-annotator training image
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1, 1.4]})
if STEM in IMAGE and os.path.getsize(IMAGE[STEM]) > 0:
    crop = np.asarray(Image.open(IMAGE[STEM]).convert("L"))[Y0:Y0 + S, X0:X0 + S]
    ax0.imshow(crop, cmap="gray", vmin=np.percentile(crop, 0.5), vmax=np.percentile(crop, 99.9))
    for i, (color, aset) in enumerate(zip((BLUE, ORANGE, AQUA), SETS_OF[STEM]), 1):
        masks = mu.decode(R[aset])[Y0:Y0 + S, X0:X0 + S]
        for j in range(masks.shape[2]):
            if masks[:, :, j].any():
                ax0.contour(masks[:, :, j], levels=[0.5], colors=[color], linewidths=1.3)
        ax0.plot([], [], color=color, lw=2, label=f"annotator {i}")
    ax0.legend(loc="upper right", frameon=True, facecolor="black", framealpha=0.55, edgecolor="none",
               labelcolor="white", fontsize=8.5)
ax0.set_title("One crop, three experts", loc="left")
ax0.set_axis_off()
rows = [(BIN_NAMES[i], summarize(*BY_AREA[i][:4]), BY_AREA[i][5] / BY_AREA[i][4])
        for i in range(len(BIN_NAMES)) if i in BY_AREA]
x = np.arange(len(rows))
for k, (name, color, values) in enumerate((("match rate", BLUE, [m for _, _, m in rows]),
                                            ("SQ (mean IoU of the matches)", ORANGE, [r["SQ"] for _, r, _ in rows]),
                                            ("PQ", AQUA, [r["PQ"] for _, r, _ in rows]))):
    ax1.bar(x + (k - 1) * 0.27, values, width=0.24, color=color, label=name)
ax1.set_xticks(x, [n.replace(" px", "").replace("under ", "<").replace(" and up", "+") for n, _, _ in rows])
ax1.set_xlabel("area of the labelled filament (px)")
ax1.set_ylim(0, 1)
ax1.legend(loc="upper left", ncol=3)
ax1.set_title("Expert-vs-expert agreement by filament size", loc="left")
plt.tight_layout()
plt.show()

This is not a hard ceiling: a model trained on what most annotators mark can score higher against each of them than any single annotator does. I read 0.34 as a yardstick, expert-level agreement on this metric, not as a limit.

## 4. How far a score moves by chance

The public leaderboard scores about half of the 180 test images. To see how far a score moves just from which images land in that half, I resample the multi-annotator training images with replacement, score each image as one annotator against the image's other annotator-sets, and recompute the pooled PQ 2,000 times for 45, 90 and 180 images. I also draw two 90-image halves at a time, as the public and private boards are. For the gap between two different submissions on the same images, I use the three-annotator images: each annotator-set is the ground truth in turn, the other two annotators are split into predictor X and predictor Y, and I bootstrap the gap between their pooled PQs.

In [ ]:
def counts(pred, truth):
    """(IoU sum, TP, FP, FN) of one annotator-set scored as a prediction against another."""
    iou = np.asarray(mu.iou(R[pred], R[truth], [0] * len(R[truth]))).T
    hit = iou > 0.5
    return np.array([iou[hit].sum(), hit.sum(), (hit.sum(0) == 0).sum(), (hit.sum(1) == 0).sum()], float)


def pq_of(c):
    return c[0] / (c[1] + 0.5 * c[2] + 0.5 * c[3])


# Each image counts as one predictor scored against the image's other annotator-sets: the counts of all ordered
# pairs of the image, divided by its number of annotator-sets (so a three-annotator image weighs twice as much as
# a two-annotator one, not three times as in the pooled number of section 3).
PER_IMAGE = np.array([sum(counts(a, b) for a, b in itertools.permutations(SETS_OF[s], 2)) / len(SETS_OF[s])
                      for s in MULTI])
rs = np.random.RandomState(0)                       # the legacy generator draws the same on every numpy version
BOOT = {n: np.array([pq_of(PER_IMAGE[rs.randint(0, len(PER_IMAGE), n)].sum(0)) for _ in range(2000)])
        for n in (45, 90, 180)}
print(f"pooled PQ with every image weighted as one predictor: {pq_of(PER_IMAGE.sum(0)):.4f}")
for n, v in BOOT.items():
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"{n:>4} images: SD {v.std():.4f}; 95% of draws between {lo:.3f} and {hi:.3f}")
# The public and private halves are two different draws of 90 images: their gap for the same predictions.
split = []
for _ in range(2000):
    i = rs.randint(0, len(PER_IMAGE), 180)
    split.append(pq_of(PER_IMAGE[i[:90]].sum(0)) - pq_of(PER_IMAGE[i[90:]].sum(0)))
print(f"same predictions, one 90-image half minus another: SD {np.std(split):.4f}")

# Two different predictors on the same images: on each three-annotator image every annotator-set is the ground
# truth in turn, and of the other two annotators the one listed first is predictor X, the other predictor Y.
THREE = [SETS_OF[s] for s in MULTI if len(SETS_OF[s]) == 3]
X = np.array([sum(counts([u for u in v if u != t][0], t) for t in v) for v in THREE])
Y = np.array([sum(counts([u for u in v if u != t][1], t) for t in v) for v in THREE])
print(f"\ntwo predictors on the same {len(THREE)} three-annotator images: PQ {pq_of(X.sum(0)):.4f} "
      f"and {pq_of(Y.sum(0)):.4f}")
for n in (45, 90, 180):
    gap = []
    for _ in range(2000):
        i = rs.randint(0, len(THREE), n)
        gap.append(pq_of(X[i].sum(0)) - pq_of(Y[i].sum(0)))
    print(f"{n:>4} images: SD of the gap between them {np.std(gap):.4f}")

BOARD = {'taken': '2026-10-12 05:30 UTC', 'teams': 972, 'p50': 0.34, 'p75': 0.37, 'p90': 0.4}   # aggregates of the public leaderboard CSV, downloaded with the Kaggle API (no team rows)
print(f"\npublic leaderboard on {BOARD['taken']}: {BOARD['teams']} teams; median {BOARD['p50']:.2f}, "
      f"75th percentile {BOARD['p75']:.2f}, 90th percentile {BOARD['p90']:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
edges = np.linspace(min(v.min() for v in BOOT.values()), max(v.max() for v in BOOT.values()), 60)
for n, color in zip(BOOT, ("#86b6ef", BLUE, "#104281")):
    ax.hist(BOOT[n], bins=edges, histtype="step", lw=2, color=color, label=f"{n} images (SD {BOOT[n].std():.3f})")
ax.axvline(pq_of(PER_IMAGE.sum(0)), color=INK2, lw=1)
ax.set_xlabel("pooled PQ of the same expert predictions on a random draw of images")
ax.set_ylabel("draws out of 2,000")
ax.set_title("How far one fixed set of predictions moves with the image draw", loc="left")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

At 90 images the SD is 0.0144: the same predictions land anywhere from 0.314 to 0.371 in 95% of draws, and even 180 images only bring the SD down to 0.0104. (Here each image counts as one predictor, which weighs a three-annotator image twice as much as a two-annotator one instead of three times, so the pooled PQ reads 0.3421 rather than 0.3399.) Two halves of the same predictions differ with an SD of 0.0205, and two different predictors scored on the same 90 images with an SD of 0.0103; the board then rounds to two decimals. So I would treat as noise a gap under about 0.03 between a CV score and the public board, under about 0.04 between the public and private boards, and under about 0.02 between two public scores, and choose between submissions on out-of-fold PQ instead.

## 5. When is a prediction worth keeping?

PQ has a simple break-even rule. One more predicted instance always adds 0.5 to the denominator: if it hits a filament that nothing else hits, a false negative (0.5) becomes a true positive (1); if it misses, it adds a false positive (0.5). It adds its IoU to the numerator only when it hits. So it raises PQ exactly when

**P(match) x E[IoU | match] > 0.5 x PQ**

with P(match) averaged over the image's annotator-sets, since the prediction is scored against each of them. This is the PQ/2 rule that \_.tushhaaarrrr derives and uses to tune the keep threshold in [Solar Filament Segmentation 2026 (0.37)](https://www.kaggle.com/code/tushhaaarrrr/solar-filament-segmentation-2026-0-37). Below I check it on the expert labels, using each annotator's filaments as the candidates.

In [ ]:
BAR = 0.5 * HVH["PQ"]
cand = pd.concat([pd.DataFrame({"area": AREA[a], "hit": (iou > 0.5).any(0), "gain": (iou * (iou > 0.5)).sum(0)})
                  for s, k, a, b, iou in PAIRS], ignore_index=True)   # each predicted filament x each other annotator
CUTS = [0, 200, 300, 400, 600, 800, 1200, 2000, np.inf]
cand["bin"] = np.digitize(cand["area"], CUTS) - 1
print(f"keep a candidate when P(match) x E[IoU | match] > 0.5 x PQ = {BAR:.3f}, "
      f"that is P(match) > {BAR / HVH['SQ']:.2f} at SQ {HVH['SQ']:.3f}")
print(f"\n{'predicted area':<16}{'comparisons':>12}{'P(match)':>10}{'E[IoU|match]':>14}{'product':>9}{'SE':>7}  verdict")
GAIN, SE = {}, {}
for i, g in cand.groupby("bin"):
    lo, hi = CUTS[i], CUTS[i + 1]
    name = f"{lo:,}-{hi:,} px" if np.isfinite(hi) else f"{lo:,} px and up"
    p, gain = g["hit"].mean(), g["gain"].mean()
    se = g["gain"].std(ddof=1) / np.sqrt(len(g))        # standard error, comparisons treated as independent
    GAIN[name], SE[name] = gain, se
    verdict = "keep" if gain - 2 * se > BAR else "drop" if gain + 2 * se < BAR else "level"   # 2 SE from the bar
    print(f"{name:<16}{len(g):>12}{p:>10.3f}{gain / p:>14.3f}{gain:>9.3f}{se:>7.3f}  {verdict}")
SWEEP = {T: human_vs_human(MULTI, min_area=T)["PQ"] for T in (0, 100, 200, 300, 400, 600, 800, 1200, 2000)}
print("\nPQ when predicted filaments under T px are dropped (labels untouched):")
print("  " + ", ".join(f"T={T}: {v:.4f}" for T, v in SWEEP.items()))

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4))
names = list(GAIN)
worth = [GAIN[n] - 2 * SE[n] > BAR for n in names]
ax0.bar(range(len(names)), [GAIN[n] for n in names], width=0.6, color=[BLUE if w else GRAY for w in worth],
        yerr=[2 * SE[n] for n in names], error_kw={"ecolor": INK2, "elinewidth": 1, "capsize": 3})
line = ax0.axhline(BAR, color=INK, lw=1, label=f"break-even: 0.5 x PQ = {BAR:.3f}")
short = lambda n: n.replace(" px", "").replace(",000", "k").replace(",200", ".2k").replace(" and up", "+")
ax0.set_xticks(range(len(names)), [short(n) for n in names])
ax0.set_xlabel("area of the predicted filament (px)")
ax0.set_ylabel("P(match) x E[IoU | match], with 2 SE")
ax0.legend(handles=[line, Patch(color=BLUE, label="clears the bar by more than 2 SE: keep"),
                    Patch(color=GRAY, label="within 2 SE of the bar")], loc="upper left")
ax0.set_title("What one more candidate adds, by its size", loc="left")
ax1.plot(list(SWEEP), list(SWEEP.values()), color=BLUE, lw=2, marker="o", ms=5)
ax1.set_xlabel("minimum predicted area T (px)")
ax1.set_ylabel("pooled PQ")
ax1.set_title("Expert-vs-expert PQ after dropping predictions under T px", loc="left")
plt.tight_layout()
plt.show()

At the human level the bar is 0.5 x PQ = 0.170, which at the observed SQ means P(match) above 0.27. Candidates under 300 px sit within two standard errors of the bar (0.137 under 200 px on 58 comparisons, 0.170 between 200 and 300 px on 243), and every larger size clears it by more than that. The sweep agrees: dropping everything under 300 px moves PQ from 0.3399 to 0.3402, a wash, while every larger cut-off removes candidates that clear the bar and costs PQ (0.3113 at 800 px, 0.2695 at 1,200 px). The bar rises with your own PQ (a model at PQ 0.40 needs P(match) x E[IoU | match] above 0.20 from each candidate), so tune the minimum area and the keep threshold on out-of-fold PQ rather than copying these values.

## 6. From a probability map to disjoint instances

A semantic-segmentation model (a U-Net and the like) predicts a per-pixel probability and then needs instances. The next cell is written to be pasted into a submission notebook: `instances_from_prob` (hysteresis thresholds, 8-connected components and a minimum area; the defaults are starting points to tune on out-of-fold PQ), an RLE encoder that works from pixel coordinates instead of building one full-frame mask per instance, and `validate_submission`, checks to run on a file before uploading it. The cell after it tests them on synthetic maps only.

In [ ]:
# ===== Paste-ready: probability map -> disjoint instances -> submission rows -> checks before uploading =====
import numpy as np, pandas as pd
from scipy import ndimage as ndi
from pycocotools import mask as mu
H = W = 2048


def instances_from_prob(prob, t_hi=0.5, t_lo=0.35, min_area=250):
    """Hysteresis + 8-connected components: keeps each region above t_lo that contains a pixel above t_hi and has
    at least min_area pixels. Returns a label map (0 = background), so the instances cannot overlap.
    The defaults are starting points: tune all three on out-of-fold PQ."""
    lab, n = ndi.label(prob > t_lo, structure=np.ones((3, 3), bool))
    keep = np.zeros(n + 1, bool)
    keep[np.unique(lab[prob > t_hi])] = True
    keep &= np.bincount(lab.ravel(), minlength=n + 1) >= min_area
    keep[0] = False
    relabel = np.zeros(n + 1, np.int32)
    relabel[keep] = np.arange(1, int(keep.sum()) + 1)
    return relabel[lab]


def string_from_counts(runs):
    """Run lengths -> COCO counts string (pycocotools' rleToString, in Python)."""
    out = []
    for i in range(len(runs)):
        x = int(runs[i]) - (int(runs[i - 2]) if i > 2 else 0)
        more = True
        while more:
            c = x & 0x1F
            x >>= 5
            more = x != -1 if c & 0x10 else x != 0
            out.append(chr((c | 0x20 if more else c) + 48))
    return "".join(out)


def counts_from_string(s):
    """COCO counts string -> run lengths (pycocotools' rleFrString, in Python)."""
    runs, p = [], 0
    while p < len(s):
        x = k = 0
        more = True
        while more:
            c = ord(s[p]) - 48
            x |= (c & 0x1F) << (5 * k)
            more = bool(c & 0x20)
            p += 1
            k += 1
            if not more and c & 0x10:
                x |= -1 << (5 * k)
        runs.append(x + (runs[-2] if len(runs) > 2 else 0))
    return runs


def rle_from_pixels(ys, xs):
    """The pixels of one instance -> its submission string: runs down the columns, starting with a zero run."""
    idx = np.unique(np.asarray(xs, np.int64) * H + np.asarray(ys, np.int64))
    cut = np.flatnonzero(np.diff(idx) > 1)
    starts, ends = np.r_[idx[0], idx[cut + 1]], np.r_[idx[cut], idx[-1]] + 1
    runs = np.empty(2 * len(starts) + 1, np.int64)
    runs[0::2] = np.r_[starts[0], starts[1:] - ends[:-1], H * W - ends[-1]]
    runs[1::2] = ends - starts
    return string_from_counts(runs if runs[-1] else runs[:-1])


def rles_from_labels(lab, y0=0, x0=0):
    """Label map (0 = background) -> one submission string per instance, in label order, without building a
    full-frame mask per instance. (y0, x0) is the offset of a crop inside the 2048 x 2048 frame."""
    ys, xs = np.nonzero(lab)
    if not len(ys):                                     # nothing above the thresholds: no rows for this image
        return []
    ids = lab[ys, xs]
    order = np.argsort(ids, kind="stable")
    ys, xs, ids = ys[order] + y0, xs[order] + x0, ids[order]
    cuts = np.flatnonzero(np.diff(ids)) + 1
    return [rle_from_pixels(y, x) for y, x in zip(np.split(ys, cuts), np.split(xs, cuts))]


def submission_rows(stem, lab):
    """Rows for one image: filament_id '<stem>_<k>' and the instance's string. lab must be at full resolution."""
    assert lab.shape == (H, W), f"{stem}: the label map is {lab.shape}; upsample to (2048, 2048) before encoding"
    return [(f"{stem}_{k}", s) for k, s in enumerate(rles_from_labels(lab), 1)]


def validate_submission(df, test_stems):
    """Checks to run on a file before uploading it. Returns (errors, warnings): no errors means ready."""
    errors, warnings = [], []
    if list(df.columns) != ["filament_id", "segmentation_rle"]:
        return [f"the columns must be filament_id, segmentation_rle (found {list(df.columns)})"], warnings
    ids, strings = df["filament_id"].fillna("").astype(str), df["segmentation_rle"].fillna("").astype(str)
    stems = ids.str.split("_", n=1).str[0]
    if ids.duplicated().any():
        errors.append(f"{int(ids.duplicated().sum())} duplicated filament_id values")
    bad = ~ids.str.contains("_", regex=False) | ~stems.isin(set(test_stems))
    if bad.any():
        errors.append(f"{int(bad.sum())} ids are not '<test image stem>_<k>', e.g. {ids[bad].iloc[0]!r}")
    area = np.full(len(df), -1)
    for i, s in enumerate(strings):
        try:
            c = counts_from_string(s)
            if s and "0" <= min(s) and max(s) <= "o" and min(c) >= 0 and sum(c) == H * W:
                area[i] = sum(c[1::2])
        except (IndexError, TypeError, ValueError):
            pass
    if (area < 0).any():
        errors.append(f"{int((area < 0).sum())} strings are missing or are not valid 2048 x 2048 COCO counts")
    if (area == 0).any():
        warnings.append(f"{int((area == 0).sum())} masks are empty: each one counts as a false positive")
    if (area > 0.01 * H * W).any():   # the largest training filament covers under 1% of the frame
        warnings.append(f"{int((area > 0.01 * H * W).sum())} masks cover more than 1% of the frame (inverted?)")
    overlapping = []
    for stem, idx in pd.Series(np.flatnonzero(area > 0)).groupby(stems[area > 0].to_numpy()):
        objs = [{"size": [H, W], "counts": strings.iloc[i].encode("ascii")} for i in idx]
        if len(objs) > 1 and mu.area(mu.merge(objs)) < area[idx.to_numpy()].sum():
            overlapping.append(stem)
    if overlapping:
        errors.append(f"{len(overlapping)} images have overlapping instances (the host does not accept overlapping "
                      f"predictions), e.g. {overlapping[0]}")
    missing = set(test_stems) - set(stems)
    if missing:
        warnings.append(f"{len(missing)} of {len(set(test_stems))} test images have no rows: all their filaments "
                        f"count as false negatives")
    return errors, warnings

In [ ]:
# Self-test of the cell above on synthetic maps only (a submission notebook does not need this cell).
p = np.zeros((H, W), np.float32)
p[100:140, 100:300] = 0.4
p[110:115, 120:125] = 0.9                                      # a weak body with a strong seed: kept
p[500:510, 500:510] = 0.9                                      # a strong blob of 100 px: under min_area
p[900:930, 900:1000] = 0.4                                     # a weak region with no seed: dropped
p[1500:1520, 1500:1600] = 0.8
p[1521:1540, 1500:1600] = 0.8                                  # two bodies one empty row apart: two instances
lab = instances_from_prob(p)
strings = rles_from_labels(lab)
same = all(s == mu.encode(np.asfortranarray((lab == k).astype(np.uint8)))["counts"].decode()
           for k, s in enumerate(strings, 1))
back = all(string_from_counts(counts_from_string(s)) == s for s in strings)
empty = submission_rows("blank", instances_from_prob(np.zeros((H, W), np.float32)))
assert lab.max() == 3 and same and back and empty == []
print(f"synthetic map: {lab.max()} instances, areas {np.bincount(lab.ravel())[1:].tolist()}; strings identical to "
      f"pycocotools.encode: {same}; decode and re-encode round trip: {back}; an empty map gives {len(empty)} rows")
yy, xx = np.mgrid[0:H, 0:W]
blob = lambda cx, cy, a, b: (((xx - cx) / a) ** 2 + ((yy - cy) / b) ** 2 < 1).astype(np.int32)
demo = pd.DataFrame(submission_rows(TEST_STEMS[0], blob(1000, 500, 40, 10) + 2 * blob(1500, 900, 25, 15))
                    + submission_rows(TEST_STEMS[1], blob(700, 1200, 30, 12)),
                    columns=["filament_id", "segmentation_rle"])
cases = {"a synthetic file with 3 instances on 2 test images": demo,
         "the same file plus an instance overlapping another": pd.concat(
             [demo, pd.DataFrame([(f"{TEST_STEMS[0]}_9", rles_from_labels(blob(1010, 502, 40, 10))[0])],
                                 columns=demo.columns)], ignore_index=True),
         "the same file plus a row named after a training image": pd.concat(
             [demo, pd.DataFrame([(f"{next(iter(SETS_OF))}_1", demo["segmentation_rle"].iloc[0])],
                                 columns=demo.columns)], ignore_index=True)}
for name, df in cases.items():
    errors, warnings = validate_submission(df, TEST_STEMS)
    assert (not errors) == name.startswith("a synthetic file"), (name, errors)
    print(f"validator, {name}: errors {errors or 'none'}; warnings {warnings or 'none'}")

Does skipping an instance head cost anything? I merge each annotator's filaments into one mask, split it back into 8-connected components, and score the components in place of the drawn instances, against the other annotators and against their own. (\_.tushhaaarrrr's notebook checks the second of these on a sample.)

In [ ]:
same = sum(string_from_counts(counts_from_string(s)) == s for s in GT["segmentation_rle"])
print(f"the Python decoder and encoder reproduce {same} of {len(GT)} label strings exactly")


def components_of(aset):
    """One annotator's filaments merged into a single mask, then split back into 8-connected components."""
    union = mu.merge(R[aset])
    x, y, w, h = (int(v) for v in mu.toBbox(union))
    lab, _ = ndi.label(mu.decode(union)[y:y + h, x:x + w], structure=np.ones((3, 3), bool))
    return rles_from_labels(lab, y, x)


CC = pd.DataFrame([(f"{a}_{k}", s, a, a.split("-", 1)[1]) for a in R for k, s in enumerate(components_of(a), 1)],
                  columns=["filament_id", "segmentation_rle", "annotator_set", "stem"])
CC["area"] = [int(mu.area(r)) for r in rles(CC["segmentation_rle"])]   # one at a time: area() of a long list
                                                                        # overflows a uint8 under numpy 2
shared = merged = 0
for a, g in CC.groupby("annotator_set", sort=False)["segmentation_rle"]:
    touch = np.asarray(mu.iou(rles(g), R[a], [0] * len(R[a]))).reshape(len(g), len(R[a])) > 0   # component x filament
    shared += int(touch[touch.sum(1) > 1].any(0).sum())
    merged += int(len(g) < len(R[a]))
print(f"{len(GT)} filaments -> {len(CC)} connected components; {shared} filaments ({shared / len(GT):.2%}) share a "
      f"component with another; {merged} annotator-sets have fewer components than filaments")


def self_score(table):
    """Each annotator-set's predictions scored against that same annotator's filaments, on every image."""
    total = np.zeros(4)
    for r in range(3):
        users = [v[r] for v in SETS_OF.values() if len(v) > r]
        total += pq_counts(GT[GT["annotator_set"].isin(users)],
                           as_prediction(table[table["annotator_set"].isin(users)]))
    return summarize(*total)


cc = human_vs_human(MULTI, table=CC)
print(f"scored against the other annotators: drawn instances {show(HVH)}")
print(f"                                     components      {show(cc)}")
print(f"PQ lost by using connected components: {HVH['PQ'] - cc['PQ']:.4f}")
print(f"components scored against their own annotator: {show(self_score(CC))}")

The 8,199 filaments become 8,219 components, and only 20 filaments (0.24%) share a component with another. Scored against the other annotators, the components get PQ 0.3391 against 0.3399 for the drawn instances, a loss of 0.0008; scored against their own annotator they get 0.9972. So for masks as clean as the annotators' own, connected components lose about a thousandth of PQ and an instance head has very little to win. A model's masks may merge neighbouring filaments more often than an annotator's, so check yours: section 7 converts out-of-fold maps the same way before scoring them.

## 7. Score your own predictions

Set `MY_PREDICTIONS` to your out-of-fold predictions for training images, either a CSV in the submission format or a folder of per-image maps, and `MY_IMAGES` to the images they cover if your model predicted nothing on some of them. The cell scores them the way the leaderboard would, every annotator-set of an image against all your predictions for that image, pooled, and adds SQ, RQ and a bootstrap SD over your images. Your folds should be grouped by image: if one annotator-set of an image is in the training fold and another in the validation fold, the score comes out optimistic.

With `MY_PREDICTIONS = None` the cell runs a demonstration: the connected components of the first annotator on each multi-annotator image, scored against the other annotators of that image.

In [ ]:
# ===== Score your own out-of-fold predictions: the only cell to change =====
MY_PREDICTIONS = None   # a CSV in the submission format for TRAINING images, or a folder of per-image maps named
                        # <stem>.npy or <stem>.png (instance labels, a 0/1 or 0/255 mask, or .npy probabilities)
MY_IMAGES = None        # the stems to score, e.g. your validation fold; None = every image in MY_PREDICTIONS


def load_predictions(path):
    """A submission-format CSV, or a folder of per-image maps turned into instances with section 6's helpers."""
    if os.path.isfile(path):
        return pd.read_csv(path, dtype=str)[["filament_id", "segmentation_rle"]]
    rows = []
    for f in sorted(glob.glob(os.path.join(path, "*.npy")) + glob.glob(os.path.join(path, "*.png"))):
        m = np.load(f) if f.endswith(".npy") else np.asarray(Image.open(f))
        assert m.shape == (H, W), f"{f}: expected one 2048 x 2048 channel, found shape {m.shape}"
        if m.dtype.kind == "f" and 0 <= m.min() and m.max() <= 1:
            lab = instances_from_prob(m)                                  # probabilities
        elif set(np.unique(m).tolist()) <= {0, 1, 255}:
            lab = ndi.label(m > 0, structure=np.ones((3, 3), bool))[0]   # a 0/1 (or 0/255 PNG) mask
        else:
            lab = np.rint(m).astype(np.int32)                             # instance labels (any dtype)
        rows += submission_rows(os.path.splitext(os.path.basename(f))[0], lab)
    return pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])


def score_predictions(pred, stems=None, truth=None, draws=1000):
    """Training-image predictions scored the way the leaderboard scores a submission: every annotator-set of each
    image against all predictions for that image, pooled. Adds a bootstrap SD over the images scored."""
    truth = GT if truth is None else truth
    img = pred["filament_id"].str.split("_", n=1).str[0]
    stems = sorted(set(img) if stems is None else set(stems))
    unknown = sorted(set(stems) - set(SETS_OF))
    assert not unknown, f"{len(unknown)} stems are not training images, e.g. {unknown[0]}"
    truth = truth[truth["stem"].isin(stems)]
    res = host_pq(truth, pred[img.isin(stems)])
    t_by = dict(tuple(truth.groupby("stem")))
    p_by = dict(tuple(pred.groupby(img)))
    per = np.array([pq_counts(t_by[s], p_by.get(s, pred.iloc[:0])) for s in stems if s in t_by])
    rs = np.random.RandomState(0)
    sd = np.std([summarize(*per[rs.randint(0, len(per), len(per))].sum(0))["PQ"] for _ in range(draws)])
    n_sets = collections.Counter(len(SETS_OF[s]) for s in stems)
    print(f"  {len(stems)} images (" + ", ".join(f"{n_sets[k]} with {k}" for k in sorted(n_sets))
          + f" annotator-sets), {truth['annotator_set'].nunique()} annotator-sets scored")
    print(f"  {show(res)}; bootstrap SD over these images {sd:.4f}")
    return res


if MY_PREDICTIONS is None:
    firsts = [SETS_OF[s][0] for s in MULTI]
    print("demonstration: the first annotator's connected components on each multi-annotator image, scored against "
          "the other annotators of that image")
    score_predictions(as_prediction(CC[CC["annotator_set"].isin(firsts)]), MULTI,
                      truth=GT[~GT["annotator_set"].isin(firsts)])
else:
    mine = load_predictions(MY_PREDICTIONS)
    errors, _ = validate_submission(mine, list(SETS_OF))
    print(f"{MY_PREDICTIONS}: {len(mine)} predicted instances; file errors: {errors or 'none'}")
    score_predictions(mine, MY_IMAGES)
print(f"\ntotal run time {(time.time() - T0) / 60:.1f} minutes")

The demonstration lands where section 3 says it should: PQ 0.3394 with a bootstrap SD of 0.0082 over its 296 images.

## 8. What this does not claim

- The agreement and noise numbers come from training images with two or three annotator-sets. How many annotator-sets each test image has is not published, so the test-set equivalents may differ somewhat.
- Expert-vs-expert PQ is a yardstick, not an upper bound: a model that predicts what most annotators would draw can score higher against each of them than any one annotator does.
- The bootstrap assumes the public half is a random half of the test images and uses the experts as predictors. A model whose errors differ from theirs will have a somewhat different SD.
- The cost of connected components is measured on the annotators' own masks, not on a model's.
- The scorer reproduces the host's notebook, which the host pointed to as the exact implementation of the leaderboard's PQ. The upload checks beyond the overlap rule are not published.
- On the leaderboard itself, the host has said in the forum that it will be cleaned up ([topic](https://www.kaggle.com/competitions/filament-segmentation-2026/discussion/740400)) and that high-scoring submissions will be examined ([topic](https://www.kaggle.com/competitions/filament-segmentation-2026/discussion/747399)).

## Credits

- Azim Ahmadzadeh (host) for [Self_Evaluation_Notebook](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook), the reference scorer that section 2 follows.
- \_.tushhaaarrrr for [Solar Filament Segmentation 2026 (0.37)](https://www.kaggle.com/code/tushhaaarrrr/solar-filament-segmentation-2026-0-37), which already reports three of the ideas measured here: one annotator judged by the others scores about 0.34, the connected components of an annotator's mask recover that annotator's instances, and the PQ/2 keep rule that section 5 checks.
- The labels are MAGFiLO, Manually Annotated GONG Filaments from H-Alpha Observations ([Scientific Data paper](https://www.nature.com/articles/s41597-024-03876-y)), shared for this competition under CC BY-NC 4.0. The masks use pycocotools.
- Mine: the host's exact logic on full-resolution RLE strings and its checks, the breakdowns by size and category with the count of near-miss outlines, the noise floor and the paired gap, the break-even table with its standard errors, the cost of components against the other annotators, the encoder and the validator, and the scoring cell.